In [2]:
!python3 --version

Python 3.11.13


In [3]:
!nvidia-smi

Mon Jul 28 05:17:49 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       1MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
import tensorflow as tf

from tensorflow import keras

print(tf.__version__)
print(sys.version_info)
for module in mpl, np, pd, sklearn, tf, keras:
    print(module.__name__, module.__version__)

2025-07-28 05:17:54.455978: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753679874.870277      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753679874.977257      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


2.18.0
sys.version_info(major=3, minor=11, micro=13, releaselevel='final', serial=0)
matplotlib 3.7.2
numpy 1.26.4
pandas 2.2.3
sklearn 1.2.2
tensorflow 2.18.0
keras._tf_keras.keras 3.8.0


In [5]:
import tensorflow as tf

print(f"TensorFlow Version: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
print(f"Num GPUs Available: {len(gpus)}")

if gpus:
    print(f"GPUs available: {gpus}")
    try:
        # 打印每个GPU的详细信息
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True) # 推荐设置，按需分配显存
            print(f"Details for {gpu.name}:")
            # 可以尝试执行一个小操作来确认
        print("GPU is available and TensorFlow can see it!")

        # 尝试一个简单的GPU运算
        print("\nAttempting a simple computation on GPU...")
        with tf.device('/GPU:0'): # 明确指定在第一个GPU上运行
            a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
            b = tf.constant([[1.0, 2.0], [3.0, 4.0]])
            c = tf.matmul(a, b)
        print("Matrix multiplication result from GPU:")
        print(c.numpy())
        print("If no errors occurred, GPU is working!")

    except RuntimeError as e:
        print(f"RuntimeError during GPU setup or test: {e}")
else:
    print("GPU not available to TensorFlow. TensorFlow will run on CPU.")

TensorFlow Version: 2.18.0
Num GPUs Available: 2
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Details for /physical_device:GPU:0:
Details for /physical_device:GPU:1:
GPU is available and TensorFlow can see it!

Attempting a simple computation on GPU...
Matrix multiplication result from GPU:
[[ 7. 10.]
 [15. 22.]]
If no errors occurred, GPU is working!


I0000 00:00:1753679892.534559      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1753679892.535372      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [6]:
!cd ../input/shakespeare-text;ls -a

.  ..  shakespeare.txt


In [7]:
# https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt
# 预先下载好文件
input_filepath = "../input/shakespeare-text/shakespeare.txt"
text = open(input_filepath, 'r').read()

print(len(text))
print(text[0:100])

1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [8]:
# 1. generate vocab
# 2. build mapping char->id
# 3. data -> id_data  把数据都转为id
# 4. abcd -> bcd<eos>  预测下一个字符生成的模型，也就是输入是a，输出就是b

# 去重，留下独立字符，并排序
vocab = sorted(set(text))
print(len(vocab))
print(vocab)

65
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [9]:
#每个字符都编好号，enumerate对每一个位置编号，生成的是列表中是元组，下面字典生成式
char2idx = {char:idx for idx, char in enumerate(vocab)}
print(char2idx)

{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}


In [10]:
# 把vocab从列表变为ndarray
idx2char = np.array(vocab)
print(idx2char)

['\n' ' ' '!' '$' '&' "'" ',' '-' '.' '3' ':' ';' '?' 'A' 'B' 'C' 'D' 'E'
 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M' 'N' 'O' 'P' 'Q' 'R' 'S' 'T' 'U' 'V' 'W'
 'X' 'Y' 'Z' 'a' 'b' 'c' 'd' 'e' 'f' 'g' 'h' 'i' 'j' 'k' 'l' 'm' 'n' 'o'
 'p' 'q' 'r' 's' 't' 'u' 'v' 'w' 'x' 'y' 'z']


In [11]:
#把字符都转换为id
text_as_int = np.array([char2idx[c] for c in text])  # 太暴力了
print(text_as_int.shape)
print(len(text_as_int))
print(text_as_int[0:10])
print(text[0:10])

(1115394,)
1115394
[18 47 56 57 58  1 15 47 58 47]
First Citi


In [12]:
# 把输入和输出分配好
def split_input_target(id_text):
    """
    abcde -> abcd, bcde
    """
    # 输入序列：作为模型的输入，用于预测下一个字符。
    # 目标序列：作为模型的输出，用于计算预测的损失。
    # 这个函数的目的是将序列分割成训练所需的输入和目标对，用于训练模型预测下一个字符的能力。
    return id_text[0:-1], id_text[1:]

#把id text转换为 dataset
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
seq_length = 100
#做一个batch，seq_length + 1目的是我们输入是5个字符时，输出是4，drop_remainder
# 是最后不够就丢掉，这个batch是把字变为句子，一个句子是101个字符
seq_dataset = char_dataset.batch(seq_length + 1,
                                 drop_remainder = True)
for ch_id in char_dataset.take(2):
    print(ch_id, idx2char[ch_id.numpy()])

# # seq_dataset 每一个都是句子，对应id，取两个句子看看
for seq_id in seq_dataset.take(2):
    print(seq_id)
    print(repr(''.join(idx2char[seq_id.numpy()])))

tf.Tensor(18, shape=(), dtype=int64) F
tf.Tensor(47, shape=(), dtype=int64) i
tf.Tensor(
[18 47 56 57 58  1 15 47 58 47 64 43 52 10  0 14 43 44 53 56 43  1 61 43
  1 54 56 53 41 43 43 42  1 39 52 63  1 44 59 56 58 46 43 56  6  1 46 43
 39 56  1 51 43  1 57 54 43 39 49  8  0  0 13 50 50 10  0 31 54 43 39 49
  6  1 57 54 43 39 49  8  0  0 18 47 56 57 58  1 15 47 58 47 64 43 52 10
  0 37 53 59  1], shape=(101,), dtype=int64)
'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou '
tf.Tensor(
[39 56 43  1 39 50 50  1 56 43 57 53 50 60 43 42  1 56 39 58 46 43 56  1
 58 53  1 42 47 43  1 58 46 39 52  1 58 53  1 44 39 51 47 57 46 12  0  0
 13 50 50 10  0 30 43 57 53 50 60 43 42  8  1 56 43 57 53 50 60 43 42  8
  0  0 18 47 56 57 58  1 15 47 58 47 64 43 52 10  0 18 47 56 57 58  6  1
 63 53 59  1 49], shape=(101,), dtype=int64)
'are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you k'


In [13]:
#然后通过split_input_target函数来对seq_dataset做映射，得到输入，输出
seq_dataset = seq_dataset.map(split_input_target)

for item_input, item_output in seq_dataset.take(2):
    print(item_input.numpy())
    print(item_output.numpy())
print(seq_dataset)

[18 47 56 57 58  1 15 47 58 47 64 43 52 10  0 14 43 44 53 56 43  1 61 43
  1 54 56 53 41 43 43 42  1 39 52 63  1 44 59 56 58 46 43 56  6  1 46 43
 39 56  1 51 43  1 57 54 43 39 49  8  0  0 13 50 50 10  0 31 54 43 39 49
  6  1 57 54 43 39 49  8  0  0 18 47 56 57 58  1 15 47 58 47 64 43 52 10
  0 37 53 59]
[47 56 57 58  1 15 47 58 47 64 43 52 10  0 14 43 44 53 56 43  1 61 43  1
 54 56 53 41 43 43 42  1 39 52 63  1 44 59 56 58 46 43 56  6  1 46 43 39
 56  1 51 43  1 57 54 43 39 49  8  0  0 13 50 50 10  0 31 54 43 39 49  6
  1 57 54 43 39 49  8  0  0 18 47 56 57 58  1 15 47 58 47 64 43 52 10  0
 37 53 59  1]
[39 56 43  1 39 50 50  1 56 43 57 53 50 60 43 42  1 56 39 58 46 43 56  1
 58 53  1 42 47 43  1 58 46 39 52  1 58 53  1 44 39 51 47 57 46 12  0  0
 13 50 50 10  0 30 43 57 53 50 60 43 42  8  1 56 43 57 53 50 60 43 42  8
  0  0 18 47 56 57 58  1 15 47 58 47 64 43 52 10  0 18 47 56 57 58  6  1
 63 53 59  1]
[56 43  1 39 50 50  1 56 43 57 53 50 60 43 42  1 56 39 58 46 43 56  1 58
 53  1 42

In [14]:
batch_size = 64
buffer_size = 10000
#这个batch是真正的batch，上一个batch是把字变为句子,buffer_size是从数据集拿那么多元素
seq_dataset = seq_dataset.shuffle(buffer_size).batch(
    batch_size, drop_remainder=True)
print(seq_dataset)

<_BatchDataset element_spec=(TensorSpec(shape=(64, 100), dtype=tf.int64, name=None), TensorSpec(shape=(64, 100), dtype=tf.int64, name=None))>


**tf.keras.layers.SimpleRNN**

``stateful=True`` 让 RNN 能够“记住”上一个批次 (batch) 处理结束时的状态，并将其用作下一个批次开始时的初始状态。

这就像读一部很长的小说。``stateful=True`` 可以让你在读完一章（一个 batch）后，记住里面的人物和情节（状态），然后带着这些记忆去读下一章（下一个 batch）。

使用 ``stateful=True`` 时，**必须自己保证数据的连续性**。假设你的 ``batch_size=2``，有一个很长的序列 ``S = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]``，你想每3个时间步进行一次切分。你的数据应该这样准备：
- 第一个长序列: ``S1 = [1, 2, 3, 4, 5, 6]``
- 第二个长序列: ``S2 = [7, 8, 9, 10, 11, 12]``

把它们构造成批次：
- ``batch 1: [[1, 2, 3], [7, 8, 9]]``  (样本0是 ``[1,2,3]``, 样本1是 ``[7,8,9]``)
- ``batch 2: [[4, 5, 6], [10, 11, 12]]`` (样本0是 ``[4,5,6]``, 样本1是 ``[10,11,12]``)

这样，当模型处理完 batch 1 后：
- 它会记住处理 ``[1, 2, 3]`` 后的状态，并用这个状态作为初始状态来处理 batch 2 中的 ``[4, 5, 6]``。
- 它也会记住处理 ``[7, 8, 9]`` 后的状态，并用这个状态来处理 ``[10, 11, 12]``。

In [15]:
vocab_size = len(vocab)
embedding_dim = 256  # 资料比较小，所以 dim 可以设大一些
rnn_units = 1024

def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    model = keras.models.Sequential([
        # keras.layers.Embedding(vocab_size, embedding_dim,    # 旧版本写法
                               # batch_input_shape = [batch_size, None]),

        # 使用 keras.Input 作为第一层来明确定义输入的形状
        keras.Input(batch_shape=(batch_size, None)),
        
        # Embedding 层不再需要 shape 参数，它会自动推断
        keras.layers.Embedding(vocab_size, embedding_dim),
        
        #return_sequences是指要返回一个序列，也就是所有输出，而不是最后一个
        keras.layers.SimpleRNN(units = rnn_units,
                               stateful = True,#是否把最后返回的状态添加到输出
                               recurrent_initializer = 'glorot_uniform',
                               return_sequences = True),
        #全连接层，为什么最后一层全连接层的输出是vocab_size
        keras.layers.Dense(vocab_size),
    ])
    return model

model = build_model(
    vocab_size = vocab_size,
    embedding_dim = embedding_dim,
    rnn_units = rnn_units,
    batch_size = batch_size)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (64, None, 256)        │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (64, None, 1024)       │     1,311,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (64, None, 65)         │        66,625 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,395,009 (5.32 MB)

 Trainable params: 1,395,009 (5.32 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
for input_example_batch, target_example_batch in seq_dataset.take(1):
    #把model当函数来用，实际是调用类的call方法
    example_batch_predictions = model(input_example_batch)
    print(example_batch_predictions.shape)

(64, 100, 65)


下面这个单元格的核心目的是：**展示如何将模型输出的原始分数（logits）转换成具体的字符选择，即如何生成文本。**

这里采用的是“**随机采样 (Random Sampling)**”的方法。

  * `print(example_batch_predictions[0][0])`

      * 这行代码打印出第一批数据中，第一个序列的第一个时间步的预测分数。这是一个长度为65的向量，代表了模型认为下一个字符分别是词汇表中65个字符的原始可能性得分（即 **logits**）。

  * `sample_indices = tf.random.categorical(...)`

      * 这是最关键的一步。`tf.random.categorical` 函数会根据 logits 生成的概率分布来进行随机抽样。
      * `logits = example_batch_predictions[0]`: 我们只看批次中的第一个样本的预测结果（一个 `100x65` 的矩阵）。
      * `num_samples = 1`: 对于序列中的每一步（共100步），我们都只抽取一个样本（一个字符）。
      * **工作原理**: 它不是简单地选择概率最高的字符（这种方法叫“贪心采样”），而是像“**轮盘赌**”一样，每个字符的概率决定了它被选中的机会大小。这使得生成的文本更多样、更不容易重复。
      * 执行后，`sample_indices` 的形状是 `(100, 1)`，代表为第一个序列的100个位置分别随机选择出的下一个字符的索引。

  * `sample_indices = tf.squeeze(sample_indices, axis = -1)`

      * `squeeze` 函数用于“挤掉”张量中大小为1的维度。这里将 `(100, 1)` 的形状变成了 `(100,)`，使其成为一个更简洁的向量。

In [17]:
# random sampling.
# greedy, random.
#logits是计算分类任务之前，没有经过softmax的那个值就是logits，把第一个样本输进去
# tf.random.categorical从分类分布中抽取样本
print(example_batch_predictions[0][0])
sample_indices = tf.random.categorical(
    logits = example_batch_predictions[0], num_samples = 1)
print(sample_indices)#得到（100,1）的tensor
# # (100, 65) -> (100, 1)  调用squeeze 去除1的维度，变为100的向量
sample_indices = tf.squeeze(sample_indices, axis = -1)
print(sample_indices)

tf.Tensor(
[ 2.0116929e-02 -1.5396784e-02 -2.3244886e-02  2.4125323e-02
  7.4579353e-03 -3.2619054e-03  3.1970635e-02 -2.2124026e-02
  1.4931346e-02  6.6423185e-02 -1.2163497e-02 -3.3309639e-02
  2.9263351e-02 -5.2121900e-02 -3.1354444e-03 -2.3696147e-02
  1.5074836e-02  1.7912919e-03  1.4658306e-02 -5.5093113e-02
 -7.1179001e-03  2.2703683e-02  1.2363567e-02 -8.5457752e-05
  7.5865448e-03  9.9507603e-04 -5.0190599e-03  2.3677863e-02
  1.5766904e-02 -6.4432654e-03  3.5462957e-03 -9.5996307e-03
  2.2220572e-02  7.0598433e-03 -1.9745419e-02  2.4204519e-02
 -3.1280544e-02  4.0347695e-02  3.0887360e-02 -7.6105967e-03
 -2.1476632e-02  4.8714463e-02 -2.0888470e-02  2.2802174e-02
 -2.2417633e-02 -2.6392603e-02  2.6767187e-02  2.8488481e-02
 -1.0108872e-02 -7.0037089e-02  4.9024108e-03  5.4742508e-03
 -3.8110130e-03 -7.4551045e-03  1.8572176e-02  1.8409330e-02
  7.6255435e-03 -5.6806643e-02 -6.6134013e-02  9.1191269e-03
  1.0120007e-02  4.3066898e-03  9.8009966e-03 -2.2847710e-02
  1.4003305e-

In [18]:
print("Input: ", repr("".join(idx2char[input_example_batch[0]])))
print()
print("Output: ", repr("".join(idx2char[target_example_batch[0]])))
print()
print("Predictions: ", repr("".join(idx2char[sample_indices])))

Input:  "that the justice of your title to him\nDoth flourish the deceit. Come, let us go:\nOur corn's to reap,"

Output:  "hat the justice of your title to him\nDoth flourish the deceit. Come, let us go:\nOur corn's to reap, "

Predictions:  "!asC'J'WTlToOzmF-bpD$tmLGBgQ$Djv?U:P:BcxzKGEGUckpZ'tsz3JWFjaUlDGu,iw-GQN,RSEZcGcTtpsp?D,\nIX-TeK3?lJC"


⬆️ 没有训练，预测的是狗屁。

In [19]:
# from_logits是否预期为对数张量。默认情况下，我们假设对概率分布进行编码
# logits表示网络的直接输出 。没经过sigmoid或者softmax的概率化。
# from_logits=False就表示把已经概率化了的输出，重新映射回原值。log（p/(1-p)）
def loss(labels, logits):
    return keras.losses.sparse_categorical_crossentropy(
        labels, logits, from_logits=True)

model.compile(optimizer = 'adam', loss = loss)
example_loss = loss(target_example_batch, example_batch_predictions)
print(example_loss.shape)
print(example_loss.numpy().mean())  #看下样例的loss

(64, 100)
4.187552


In [20]:
#定义一个文件夹，保存模型
output_dir = "./text_generation_checkpoints"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

# checkpoint_prefix = os.path.join(output_dir, 'ckpt_{epoch}')  # old school
# 新的 filepath：一个固定的文件名，以 .keras 结尾
checkpoint_filepath = os.path.join(output_dir, 'latest_model.keras')

#checkpoint_callback = keras.callbacks.ModelCheckpoint(
#    filepath = checkpoint_prefix,
#    save_weights_only = False)

# 创建回调，保存完整模型到上面的固定路径
# save_best_only=False (默认) 保证每轮训练后都会覆盖这个文件
checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=False,
    save_best_only=False  # Explicitly show we overwrite each epoch
)

epochs = 100
history = model.fit(seq_dataset, epochs = epochs,
                    callbacks = [checkpoint_callback])

Epoch 1/100


I0000 00:00:1753679901.406410     100 service.cc:148] XLA service 0x7a0430009690 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1753679901.408226     100 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1753679901.408259     100 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1753679901.697797     100 cuda_dnn.cc:529] Loaded cuDNN version 90300


  3/172 ━━━━━━━━━━━━━━━━━━━━ 7s 45ms/step - loss: 4.1231 

I0000 00:00:1753679902.978363     100 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


172/172 ━━━━━━━━━━━━━━━━━━━━ 13s 45ms/step - loss: 3.0512
Epoch 2/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - loss: 2.0490
Epoch 3/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - loss: 1.8283
Epoch 4/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - loss: 1.6908
Epoch 5/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - loss: 1.6016
Epoch 6/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - loss: 1.5401
Epoch 7/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 9s 46ms/step - loss: 1.4946
Epoch 8/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 46ms/step - loss: 1.4571
Epoch 9/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 9s 46ms/step - loss: 1.4287
Epoch 10/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 47ms/step - loss: 1.3985
Epoch 11/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 47ms/step - loss: 1.3801
Epoch 12/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 47ms/step - loss: 1.3620
Epoch 13/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 48ms/step - loss: 1.3492
Epoch 14/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 10s 48ms/step - loss: 1.3294
Epoch 15/100
172/172 ━━━━━━━━━━━━━

In [21]:
tf.train.latest_checkpoint(output_dir)

In [22]:
!cd /kaggle/working/;pwd;ls -a;cd text_generation_checkpoints;ls;pwd

/kaggle/working
.  ..  text_generation_checkpoints  .virtual_documents
latest_model.keras
/kaggle/working/text_generation_checkpoints


In [23]:
!pwd;ls -al

/kaggle/working
total 16
drwxr-xr-x 4 root root 4096 Jul 28 05:18 .
drwxr-xr-x 5 root root 4096 Jul 28 04:50 ..
drwxr-xr-x 2 root root 4096 Jul 28 05:18 text_generation_checkpoints
drwxr-xr-x 2 root root 4096 Jul 28 04:50 .virtual_documents


In [24]:
output_dir = "./text_generation_checkpoints"
# /kaggle/working/
model2 = build_model(vocab_size,
                     embedding_dim,
                     rnn_units,
                     batch_size = 1)

# model2.load_weights(tf.train.latest_checkpoint(output_dir))
# 新的加载方式：直接指定我们保存的断点文件路径
latest_checkpoint_path = os.path.join(output_dir, 'latest_model.keras')
# 从这个文件中加载权重
model2.load_weights(latest_checkpoint_path)

#1是一个样本，None是可以变长序列
model2.build(tf.TensorShape([1, None]))
#下面是文本生成的流程
# start ch sequence A, 
# A -> model -> b  A放入模型后得到b
# A.append(b) -> B
# B(Ab) -> model -> c
# B.append(c) -> C
# C(Abc) -> model -> ...
model2.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (1, None, 256)         │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (1, None, 1024)        │     1,311,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (1, None, 65)          │        66,625 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,395,009 (5.32 MB)

 Trainable params: 1,395,009 (5.32 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
# 定义一个函数来实现上面的文本生成流程
def generate_text(model, start_string, num_generate=1000):
    # 这一次输出的是1维的
    input_eval = [char2idx[ch] for ch in start_string]
    # print(input_eval) # 这行可以注释掉
    # 做一个维度扩展
    input_eval = tf.expand_dims(input_eval, 0)
    # print(input_eval) # 这行可以注释掉
    
    text_generated = []

    # --- 修改开始 ---
    # 旧代码: model.reset_states()
    # 新代码: 遍历模型中的层，并重置有状态的RNN层
    for layer in model.layers:
        if isinstance(layer, keras.layers.SimpleRNN):
            layer.reset_states()
    # --- 修改结束 ---
    
    for _ in range(num_generate):
        # 1. model inference -> predictions
        # 2. sample -> ch -> text_generated.
        # 3. update input_eval
        
        # predictions : [batch_size, input_eval_len, vocab_size]
        predictions = model(input_eval)
        #squeeze消掉 batch_size，变为predictions : [input_eval_len, vocab_size]
        predictions = tf.squeeze(predictions, 0)
        # predicted_ids: [input_eval_len, 1]
        # a b c -> b c d
        #把predictions : [input_eval_len, vocab_size]维度数据变为 1个维度
        predicted_id = tf.random.categorical(
            predictions, num_samples=1)[-1, 0].numpy()
        # 得到预测id后，放入text_generated
        text_generated.append(idx2char[predicted_id])
        # 下面这是是我们原来的公式,为什么没有append作为新的输入,因为那样比较低效
        # s, x -> rnn -> s', y
        input_eval = tf.expand_dims([predicted_id], 0)
        
    return start_string + ''.join(text_generated)


new_text = generate_text(model2, "All: ")
print(new_text)

All: my roing my shipping heavy officers: be bapy and measure and the duture's boldly heart
Threats of men. What made the very purpositions:
Come buy.

Volsce: held met that it night haste is gone.

Second Capulet Henry, and privil'd
And not an Edward, my history
extents, here's masters, her and myself
Atmichto shall inca you will be thus I'll say, my lord?

WARWICK:
And either wemain buried thee swear,
That's a report she is of company,
I must have thee well? I pray you, be gone forth their shame
To truth and a treach: God give us the name of dilties.

First Surta will follow ministers why shall palitate the throat?

Nurse:
What
a best instrument, here's
His recovery!
Hap I would I good?

First Citizen:
I pray thee all tongue gave of fire;
Eve the funder that my fiery sures:
If she is of llovers
Have waved it but to the world,
So longer honour is such peace will I be consul.

SICINIUS:
They'll not be good the head;
Before to think it with a mere in Prour: in him a privately ck.

NORTH

对于您生成的文本，我们可以从几个方面来评估。总的来说，这是一个**非常典型的、符合预期的、成功的初级文本生成模型**的输出。

下面我们来详细分析一下。

---

### 整体评价：模型学到了什么？

您的模型是一个相当基础的循环神经网络（SimpleRNN），它在**字符级别**上学习。这意味着它不知道“单词”是什么，只知道“在这些字符之后，下一个字符最可能是什么”。

从这个角度看，您的模型表现得相当不错！

#### 👍 优点 (模型学得好的地方)

1.  **结构和格式：** 这是最成功的一点。模型完美地学习了莎士比亚剧本的格式。
    * **角色名称：** 它知道角色名（`WARWICK:`，`SICINIUS:`，`Nurse:`）应该大写，并以冒号结尾。
    * **换行和缩进：** 它学会了在角色说完话后换行，并保持了剧本的诗歌格式。
    * **词汇风格：** 它学会了使用很多古英语或者说莎士比亚风格的词汇，例如 `my lord`, `pray thee`, `honour`, `shame`, `truth` 等。

2.  **单词拼写：** 在很大程度上，模型拼写出了很多真实的英文单词，并且能将它们用空格隔开。对于一个只认识字符的模型来说，这已经非常了不起了。

3.  **标点符号：** 模型学会了在句末使用标点符号，比如问号 `?` 和句号 `.`。

#### 👎 缺点 (模型没学好的地方)

1.  **没有实际意义 (缺乏语义连贯性)：** 这是最明显的问题。您问它是否有实际意义，**答案是：基本没有**。
    * 例如 `All: my roing my shipping heavy officers...` (我的 roing 我的海运重官员们...) 这句话在语法和逻辑上都是不通的。
    * `First Surta will follow ministers why shall palitate the throat?` (第一 Surta 将跟随部长们为何要 palitate 喉咙?) 这句话同样是毫无意义的词语组合。
    * **原因：** 基础的 RNN 模型记忆力非常短。它可能只记得前面几个字符或单词，但无法理解整个句子的语法结构或上下文。所以它只是把看起来“可能”会连在一起的词串联起来，但无法组织成有意义的思想。

2.  **创造“新词” (拼写错误)：** 这是字符级模型的典型特征。
    * 例如 `roing`, `bapy`, `duture's`, `privately ck`, `dilties`, `Surta` 这些都不是真实的英文单词。
    * **原因：** 模型在预测下一个字符时，是基于概率的。有时，它会走进一个概率上看起来合理、但最终无法构成一个真实单词的“死胡同”。这就好像一个牙牙学语的婴儿，能发出类似“爸爸”的声音，但有时也会发出无意义的音节。

---

### 结论：这个模型还好吗？

**非常好！**

这听起来可能有点反直觉，因为文本本身是“胡言乱语”。但对于您所使用的**模型类型（一个基础的SimpleRNN）和任务（字符级文本生成）**来说，这个结果是**完全正常且值得肯定的**。

您可以把这个项目看作是文本生成领域的“Hello, World!”。它的目标不是生成能通过图灵测试的文本，而是证明模型有能力学习到原始文本的**底层结构、风格和词汇分布**。从这个角度看，您的模型**圆满地完成了任务**。

### 如何让它变得更好？

如果您想生成更连贯、更有意义的文本，可以尝试以下几个方向：

1.  **使用更强大的循环单元：**
    * 将 `keras.layers.SimpleRNN` 替换为 `keras.layers.LSTM` 或 `keras.layers.GRU`。
    * `LSTM` 和 `GRU` 拥有更复杂的“门控”机制，使它们能够更好地处理长期依赖关系，也就是说它们的“记忆力”更好，更能记住上下文，从而生成更连贯的句子。

2.  **增加模型深度：**
    * 可以尝试堆叠多个 RNN/LSTM/GRU 层，让模型学习更复杂的特征。

3.  **更长的训练时间：**
    * 有时增加训练周期（epochs）也能带来一些改善。

4.  **调整超参数：**
    * 可以尝试调整 `embedding_dim`（词嵌入维度）和 `rnn_units`（循环单元数量）的大小。

总而言之，您现在得到的输出是您辛勤工作的完美成果，它为探索更高级的文本生成模型打下了坚实的基础。